In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from spektral.data import Dataset, DisjointLoader, Graph
from spektral.layers import GINConv, GlobalAvgPool
import scipy.sparse as sp
import os
from sklearn.metrics import f1_score
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.metrics import categorical_accuracy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score
from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()


os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Useful functions 

In [2]:
# Custom dataset class to convert a list of Graph objects to a Spektral dataset
class list_to_spektral_dataset(Dataset):
    def __init__(self, data, **kwargs):
        self.data = data

        super().__init__(**kwargs)
        
    def read(self):
        #print(self.data[0])
        return [Graph(x=graph.x, a=graph.a, y=graph.y) for graph in self.data]

# Function to filter graphs based on labels
def filter_label(dataset, label):
    #divide dataset based on the label[]
     
    output = []
    res = []
    
    for graph in dataset: 
        if graph.y in label:
            output.append(graph)
        else: 
            res.append(graph)
            
    output_dataset = list_to_spektral_dataset(output)
    res_dataset = list_to_spektral_dataset(res)
    return  output_dataset, res_dataset

# Function to split dataset into training and testing sets
def train_test_split(dataset, train_percentage):
    # Train/test split
    idxs = np.random.permutation(len(dataset)) 
    split = int(train_percentage * len(dataset))
    idx_tr, idx_te = np.split(idxs, [split])
    #print(bool(set(idx_tr) & set(idx_te)))
    dataset_tr, dataset_te = dataset[idx_tr], dataset[idx_te]
    
    return dataset_tr, dataset_te

# Function to subsample a dataset to a specific size
def subsample(dataset, size):
    # Train/test split
    idxs = np.random.permutation(len(dataset)) 
    split = int(size)
    idx_sample, idx_re = np.split(idxs, [split])
    dataset_sample, dataset_re= dataset[idx_sample], dataset[idx_re]
    
    return dataset_sample, dataset_re

# Function to merge two datasets
def merge_dataset(dataset1, dataset2): 
    # implemented fucntions in Dataset class of spektral graph 
    merged = dataset1. __add__(dataset2)
    return merged 

# Function to convert labels to binary format
def binary_label(dataset, flag_attack):
    
    for g in dataset: 
        #g.y = np.pad(g.y, (0, 1), 'constant')
        if flag_attack: 
            y = np.zeros((2,))
            y[1] = 1
            g.y = y
        else:
            y = np.zeros((2,))
            y[0] = 1
            g.y = y
        
    return dataset

# Function to convert labels to integer format
def convert_label_integer(dataset):
    for g in dataset: 
        #g.y = np.pad(g.y, (0, 1), 'constant')
        g.y = np.where(g.y==1.)[0][0] 
        
    return dataset

# Function to find the maximum number of nodes in the dataset
def find_max_nodes(dataset):
    value = 0
    for g in dataset: 
        #g.y = np.pad(g.y, (0, 1), 'constant')
        value = max(value, g.a.shape[0])
        
    return value

# Function to update labels in the dataset from a file
def update_labels(dataset, path):
    
    
    labels = np.load(path)
    
    if len(labels)!=len(dataset):
        print("Label size {} does not match data size{}".format(len(labels),len(dataset)))
           
    i = 0 
    for g in dataset:
        g.y=labels[i]
        i+=1
        
    return dataset
    

# Function to obtain labels from the dataset
def obtain_labels(dataset):
    
    y = []
    
    for g in dataset:
        y.append(g.y)
    
    return np.array(y)

# Function to obtain a feature matrix from the dataset
def obtain_feature_matrix(dataset):

    graph_x = []
    

    
    for g in dataset:
        graph_x.append(np.sum(g.x, axis=0))
    return np.array(graph_x)


# Function to convert a sparse matrix to a tuple representation
def sparse_to_tuple(sparse_mx):
    if not sp.isspmatrix_coo(sparse_mx):
        sparse_mx = sparse_mx.tocoo()
    coords = np.vstack((sparse_mx.row, sparse_mx.col)).transpose()
    values = sparse_mx.data
    shape = sparse_mx.shape
    return coords, values, shape


## Load CFG json files as spektral graph objects

In [ ]:
################################################################################
# Load data
################################################################################ 

class GraphData(Dataset):
    

    def __init__(self, cfg_path, **kwargs):
        self.cfg_path = cfg_path

        super().__init__(**kwargs)

    def read(self):
        
        file_list = os.listdir(self.cfg_path)
        file_list_x_y = list(filter(lambda x: '_sparse_matrix' not in x and '.npz' in x, file_list))
        
       
        output = []
        
       
        for filepath in file_list_x_y:

            #full path of node attribute and label
            fullpath = os.path.join(self.cfg_path, filepath)
            #file path of adj matrix
            filepath_sp = filepath.split('.')[0] + "_sparse_matrix.npz"
            #full path pf adj matrix
            fullpath_sp = os.path.join(self.cfg_path, filepath_sp)
    
            sparse_matrix = sp.load_npz(fullpath_sp)
            sparse_matrix = sparse_matrix.astype('float32')
            
            # If the sparse_matrix size is over 46000 by 46000 ,we skipped it since
            # we are unable to allocate that much memory for an array with that shape 
            if sparse_matrix.shape[0] > 46000: 
                continue
           
    
            data = np.load(fullpath)
            
            # Remove diagonal elements
            adj = sparse_matrix - sp.dia_matrix((sparse_matrix.diagonal()[np.newaxis, :], [0]), shape=sparse_matrix.shape)
            adj.eliminate_zeros()
            # Check that diag is zero:
            assert np.diag(adj.todense()).sum() == 0

            adj_triu = sp.triu(adj)
            adj_tuple = sparse_to_tuple(adj_triu)
            edges = adj_tuple[0]
            
                
            #important! filter out noisy data where the number of basic block is less than 10 and the number of non-self edges in the upper triangle is less than 3
            # if the graph is too large, we will run into oom problems
            if data["x"].shape[0] >=10 and edges.shape[0] >= 3 and sparse_matrix.shape[0] <=46000:
                output.append(Graph(x=data['x'], a= sparse_matrix, y=data['y']))
                
               
          

        return output




## Load malware data

### March

In [8]:
path_0 =  "../additional_data/graph_features/mb24/March/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/March/1/cfg_embeddings"

malware_march_0 = GraphData(path_0) 
malware_march_1 = GraphData(path_1) 

malware_march = merge_dataset(malware_march_0, malware_march_1)
malware_march = convert_label_integer(malware_march)
print(len(malware_march))

1487


In [9]:
label = obtain_labels(malware_march)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)


The number of unique families are: 104


### April

In [10]:
path_0 =  "../additional_data/graph_features/mb24/April/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/April/1/cfg_embeddings"

malware_april_0 = GraphData(path_0) 
malware_april_1 = GraphData(path_1) 

malware_april = merge_dataset(malware_april_0, malware_april_1)
malware_april = convert_label_integer(malware_april)
print(len(malware_april))

1071


In [11]:
label = obtain_labels(malware_april)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)




The number of unique families are: 81


### May

In [12]:
path_0 =  "../additional_data/graph_features/mb24/May/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/May/1/cfg_embeddings"

malware_may_0 = GraphData(path_0) 
malware_may_1 = GraphData(path_1) 

malware_may = merge_dataset(malware_may_0, malware_may_1)
malware_may = convert_label_integer(malware_may)
print(len(malware_may))

1475


In [13]:
label = obtain_labels(malware_may)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)



The number of unique families are: 99


### July

In [18]:
path_0 =  "../additional_data/graph_features/mb24/July/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/July/1/cfg_embeddings"

malware_july_0 = GraphData(path_0) 
malware_july_1 = GraphData(path_1) 


malware_july = merge_dataset(malware_july_0, malware_july_1)
malware_july = convert_label_integer(malware_july)
print(len(malware_july))

1566


In [19]:
label = obtain_labels(malware_july)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)




The number of unique families are: 126


### Aug

In [14]:
path_0 =  "../additional_data/graph_features/mb24/Aug/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/Aug/1/cfg_embeddings"

malware_aug_0 = GraphData(path_0) 
malware_aug_1 = GraphData(path_1) 

malware_aug = merge_dataset(malware_aug_0, malware_aug_1)
malware_aug = convert_label_integer(malware_aug)
print(len(malware_aug))

1636


In [15]:
label = obtain_labels(malware_aug)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)



The number of unique families are: 111


### Sep

In [16]:
path_0 =  "../additional_data/graph_features/mb24/Sep/0/cfg_embeddings"
path_1 =  "../additional_data/graph_features/mb24/Sep/1/cfg_embeddings"

malware_sep_0 = GraphData(path_0) 
malware_sep_1 = GraphData(path_1) 

malware_sep = merge_dataset(malware_sep_0, malware_sep_1)
malware_sep = convert_label_integer(malware_sep)
print(len(malware_sep))

1300


In [17]:
label = obtain_labels(malware_sep)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print("The number of unique families are: {}".format(len(uniques)))
#print(percentages)



The number of unique families are: 92


## Load normal data source

In [35]:
path_1 =  '../data/graph_features/benign_source/dataset1/cfg_embeddings'
normal_1_source = GraphData(path_1)
normal_1_source = convert_label_integer(normal_1_source)

path_2 =  '../data/graph_features/benign_source/dataset2/cfg_embeddings'
normal_2_source = GraphData(path_2)
normal_2_source = convert_label_integer(normal_2_source)

path_3 =  '../data/graph_features/benign_source/dataset3/cfg_embeddings'
normal_3_source = GraphData(path_3)
normal_3_source = convert_label_integer(normal_3_source)

path_4 =  '../data/graph_features/benign_source/dataset4/cfg_embeddings'
normal_4_source = GraphData(path_4)
normal_4_source = convert_label_integer(normal_4_source)

source_normal = merge_dataset(normal_1_source, normal_2_source)
source_normal = merge_dataset(source_normal, normal_3_source)
source_normal = merge_dataset(source_normal, normal_4_source)
print(len(source_normal))

6510


## Load normal data target

In [36]:
path_1 =  '../data/graph_features/benign_target/dataset1/cfg_embeddings'
normal_1_target = GraphData(path_1)
normal_1_target = convert_label_integer(normal_1_target)

path_2 =  '../data/graph_features/benign_target/dataset2/cfg_embeddings'
normal_2_target = GraphData(path_2)
normal_2_target = convert_label_integer(normal_2_target)

path_3 =  '../data/graph_features/benign_target/dataset3/cfg_embeddings'
normal_3_target = GraphData(path_3)
normal_3_target = convert_label_integer(normal_3_target)

path_4 =  '../data/graph_features/benign_target/dataset4/cfg_embeddings'
normal_4_target = GraphData(path_4)
normal_4_target = convert_label_integer(normal_4_target)

target_normal = merge_dataset(normal_1_target, normal_2_target)
target_normal = merge_dataset(target_normal , normal_3_target)
target_normal = merge_dataset(target_normal , normal_4_target)
print(len(target_normal))

5768


## Build model

In [37]:

################################################################################
# Build model
################################################################################

#  Define the GIN model with specified channels and number of layers
class GIN0(Model):
    def __init__(self, channels, n_layers):
        super().__init__()
        self.conv1 = GINConv(channels, epsilon=0, mlp_hidden=[channels, channels])
        self.convs = []
        for _ in range(1, n_layers):
            self.convs.append(
                GINConv(channels, epsilon=0, mlp_hidden=[channels, channels])
            )
        self.pool = GlobalAvgPool()
        self.dense1 = Dense(channels, activation="relu")
      
        

    def call(self, inputs):
        x, a, i = inputs
        x = self.conv1([x, a])
        for conv in self.convs:
            x = conv([x, a])
        x = self.pool([x, i])
        x = self.dense1(x)
      
        return x

    
    

In [38]:
class DANN_GIN(object):
    def __init__(self,loader_source_train, 
                 loader_target_train,
                 loader_target_test, GIN, n_classes,
                 epochs=90):

        #source train and test dataset
        self.loader_source_tr = loader_source_train
        #self.loader_source_te= loader_source_test
        
        
        # Target train and test dataset
        
        self.loader_target_tr = loader_target_train
        self.loader_target_te= loader_target_test


        self.n_classes = n_classes
    
    
        self.latent_dim = 128 
        
        
        self.generator = GIN 
        self.epochs = epochs 
        

        
        #Classifier
        
        class_input = Input(shape=(128,))
        x= Dense(128, activation = "relu")(class_input)
        class_output = Dense(self.n_classes, activation = "softmax")(x)
        
        self.classifier = Model(class_input, class_output, name="classifier")
        
        #Discriminator
        
        disc_input = Input(shape=(128,))
        x = Dense(128, activation = "relu")(disc_input)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = Dense(128, activation = "relu")(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        disc_output = Dense(2, activation = "softmax")(x)
        
        self.discriminator = Model(disc_input, disc_output, name="discriminator")

        
        
      
        self.loss = tf.keras.losses.CategoricalCrossentropy()

        
        self.lr = 0.001 
        self.momentum = 0.9
        self.alpha = 0.0002

        
        
        
        self.task_optimizer=  Adam(1e-3)
        self.gen_optimizer = Adam(1e-3)
        self.disc_optimizer = Adam(1e-3)
        
        self.train_task_loss = tf.keras.metrics.Mean()
        self.train_disc_loss = tf.keras.metrics.Mean()
        self.train_gen_loss = tf.keras.metrics.Mean()
        self.train_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        self.train_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()

     
        
        self.test_target_task_loss = tf.keras.metrics.Mean()
        self.test_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        
       

        
        self.batch_size = 16

        


    def train_batch(self, x_source_train, y_source_train, x_target_train, y_target_train, epoch):
        
        
        source = np.tile([1,0], (y_source_train.shape[0], 1))
        target = np.tile([0,1], (y_target_train.shape[0], 1))
        
        target_fake = np.tile([0,1], (y_source_train.shape[0], 1))
        source_fake = np.tile([1,0], (y_target_train.shape[0], 1))

        # Train discriminator
        
        with tf.GradientTape() as disc_tape:
            y_domain_pred_source = self.discriminator(self.generator(x_source_train, training=True), training=True)
            y_domain_pred_target = self.discriminator(self.generator(x_target_train, training=True), training=True)
            
            disc_loss = self.loss(source, y_domain_pred_source) +  self.loss(target, y_domain_pred_target)  
            
        disc_grad = disc_tape.gradient(disc_loss, self.discriminator.trainable_variables)  
        self.disc_optimizer.apply_gradients(zip(disc_grad, self.discriminator.trainable_variables))
        self.train_disc_loss(disc_loss)

        # Train generator and classifier

        with tf.GradientTape() as task_tape, tf.GradientTape() as gen_tape:
            
            #Forward pass
            y_class_pred_source = self.classifier(self.generator(x_source_train, training=True), training=True)
            y_class_pred_target = self.classifier(self.generator(x_target_train, training=True), training=True)
            y_domain_pred_source = self.discriminator(self.generator(x_source_train, training=True), training=True)
            y_domain_pred_target = self.discriminator(self.generator(x_target_train, training=True), training=True)
            
            
            task_loss = self.loss(y_target_train, y_class_pred_target) + 0.5*self.loss(y_source_train, y_class_pred_source)  
            adv_loss = self.loss(target_fake, y_domain_pred_source) +  self.loss(source_fake, y_domain_pred_target)   
            gen_loss = task_loss +  adv_loss*0.1
            
           
        
         # Compute gradients   
        task_grad = task_tape.gradient(task_loss, self.classifier.trainable_variables)
        gen_grad = gen_tape.gradient(gen_loss, self.generator.trainable_variables)
        
        # Update weights 
        self.task_optimizer.apply_gradients(zip(task_grad, self.classifier.trainable_variables))
        self.gen_optimizer.apply_gradients(zip(gen_grad, self.generator.trainable_variables)) 
        
        
            

        self.train_task_loss(task_loss)
        self.train_task_accuracy(y_source_train, y_class_pred_source)
        self.train_target_task_accuracy(y_target_train, y_class_pred_target)
        self.train_gen_loss(gen_loss)
            
            


        return
    
    # Function to test the target task on a batch of data
    def test_batch(self, x_target_test, y_target_test):
       
        
        y_target_class_pred = self.classifier(self.generator(x_target_test, training=False), training=False)
        
            
    
        self.test_target_task_loss(y_target_test, y_target_class_pred)
        self.test_target_task_accuracy(y_target_test, y_target_class_pred)
        
        
        return 
    
    # Function to evaluate the model on a data loader
    def evaluate(self, loader):
        output = []
        loss_fn = tf.keras.losses.CategoricalCrossentropy()
        step = 0
        while step < loader.steps_per_epoch:
            step += 1
            inputs, target = loader.__next__()
            pred = self.classifier(self.generator(inputs, training=False), training=False)
            outs = (
                loss_fn(target, pred),
                tf.reduce_mean(categorical_accuracy(target, pred)),
                len(target),  # Keep track of batch size
            )          
            output.append(outs)
            if step == loader.steps_per_epoch:
                output = np.array(output)
                return np.average(output[:, :-1], 0, weights=output[:, -1])

      
            


    # Function to log training metrics
    def log_train(self):
        
        
        log_format = 'C_loss train: {:.4f}, Acc train source: {:.2f} , Acc train target: {:.2f}\n'+'D_loss train: {:.4f}, G_loss train: {:.4f}'

        message = log_format.format(
                 self.train_task_loss.result(),
                 self.train_task_accuracy.result()*100,
                 self.train_target_task_accuracy.result()*100,
                 self.train_disc_loss.result(),
                 self.train_gen_loss.result())
        

        self.reset_metrics('train')
      

        return message 
    # Function to log test metrics
    def log_test(self):
        
        
        log_format = "C_loss test target: {:.4f}, Acc test target: {:.2f}"

        message = log_format.format(
               
                 self.test_target_task_loss.result(),
                 self.test_target_task_accuracy.result()*100)

        self.reset_metrics('test')


        return message 
    # Function to reset metrics for training or testing
    def reset_metrics(self, target):

        if target == 'train':
            self.train_task_loss.reset_states()
            self.train_task_accuracy.reset_states()
            self.train_disc_loss.reset_states()
            self.train_gen_loss.reset_states()
    
        
        
        if target == 'test':
            self.test_target_task_loss.reset_states()
            self.test_target_task_accuracy.reset_states()
        

        return 
    # Function to train the model
    def train(self):
        epoch = step = 0
        

        # Iterate over the source and target training data
        for (source_batch, source_labels), (target_batch, target_labels) in zip(self.loader_source_tr, self.loader_target_tr):
            step +=1 
            self.train_batch(source_batch, source_labels, target_batch, target_labels, epoch)
            if step == min(self.loader_source_tr.steps_per_epoch, self.loader_target_tr.steps_per_epoch):
                step = 0
                epoch +=1  
                if epoch % 10 ==0:
                    print('Epoch: {}'.format(epoch))
                    print(self.log_train())                 
                    results_te = self.evaluate(self.loader_target_te)
                    print("Test results - Loss: {:.3f} - Acc: {:.3f}".format(*results_te))
                    
                
        return self.generator, self.classifier
                


        
                
            
            

## Training 

### target train july, target test aug

In [42]:

# source malware is the malware from March, April and May
source_malware = merge_dataset(malware_march, malware_april)
source_malware = merge_dataset(source_malware,  malware_may)

# target malware is the malware from July (train) and August (test) 
target_malware_train = malware_july 
target_malware_test = malware_aug 



# convert it to binary labels
target_malware_train = binary_label(target_malware_train, True)
target_malware_test = binary_label(target_malware_test, True)
source_malware  = binary_label(source_malware, True)
target_normal = binary_label(target_normal, False)
source_normal = binary_label(source_normal, False)

print("Target malware train size: {}".format(len(target_malware_train)))
print("Target malware test size: {}".format(len(target_malware_test)))
print("Source malware size: {}".format(len(source_malware)))
print("Source normal size: {}".format(len(source_normal)))
           
           

Target malware train size: 1566
Target malware test size: 1636
Source malware size: 4033
Source normal size: 6510


In [43]:
# Split the target normal dataset into training and testing sets
target_normal_train, target_normal_ = train_test_split(target_normal, 0.33)
target_normal_test, _ = train_test_split(target_normal_, 0.5)
print(len(target_normal_train))
print(len(target_normal_test))

1903
1932


In [44]:
# Merge the source normal and source malware datasets
source = merge_dataset(source_normal, source_malware)

# Split the source dataset into training and testing sets
source_train, source_test = train_test_split(source, 0.75)
# Construct target training and testing sets 
target_train = merge_dataset(target_normal_train, target_malware_train)
target_test = merge_dataset(target_normal_test, target_malware_test)



print("Target train dataset size: {}".format(len(target_train)))
print("Target test dataset size: {}".format(len(target_test)))
print("Source train dataset size: {}".format(len(source_train)))
print("Source test dataset size: {}".format((len(source_test))))

Target train dataset size: 3469
Target test dataset size: 3568
Source train dataset size: 7907
Source test dataset size: 2636


In [46]:
samples = [20, 50, 100, 200, 300, 500]

channels = 128  # Hidden units
layers = 3  # GIN layers
epochs = 80  # Number of training epochs
batch_size = 16  # Batch size
n_out = target_train.n_labels


for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))


    target_train_select, target_train_re = subsample(target_train, size)

    loader_source_tr = DisjointLoader(source_train, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_source_te = DisjointLoader(source_test,  batch_size=batch_size)


    loader_target_tr = DisjointLoader(target_train_select, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_target_te = DisjointLoader(target_test, batch_size=batch_size)

    # build model
    GIN = GIN0(channels, layers)


    model =DANN_GIN(loader_source_tr, loader_target_tr, loader_target_te, GIN, n_out, epochs==80)


    G, C = model.train()
    
    
    ################################################################################
    # Evaluate model
    ################################################################################
    results = []
    step = 0
    
    while step < loader_target_te.steps_per_epoch:      
        step += 1
        inputs, target = loader_target_te.__next__()
        pred = C(G(inputs, training=False), training=False)
        results.append(
            (
                f1_score(np.argmax(target, axis=1), np.argmax(pred, axis=1), average='weighted')
            )
        )
    print("Done. Test f1: {}".format(np.mean(results, 0)))
        
        
       

--------------------------Sample size 20-------------------------
Epoch: 10
C_loss train: 0.7544, Acc train source: 69.06 , Acc train target: 73.00
D_loss train: 1.6907, G_loss train: 0.9171
Test results - Loss: 0.765 - Acc: 0.567
Epoch: 20
C_loss train: 0.6093, Acc train source: 77.50 , Acc train target: 80.50
D_loss train: 1.5507, G_loss train: 0.7647
Test results - Loss: 0.804 - Acc: 0.643
Epoch: 30
C_loss train: 0.5536, Acc train source: 77.50 , Acc train target: 84.33
D_loss train: 1.5031, G_loss train: 0.7066
Test results - Loss: 0.948 - Acc: 0.693
Epoch: 40
C_loss train: 0.3694, Acc train source: 85.62 , Acc train target: 87.38
D_loss train: 1.4625, G_loss train: 0.5206
Test results - Loss: 0.910 - Acc: 0.694
Epoch: 50
C_loss train: 0.2978, Acc train source: 83.44 , Acc train target: 89.60
D_loss train: 1.4537, G_loss train: 0.4509
Test results - Loss: 1.149 - Acc: 0.636
Epoch: 60
C_loss train: 0.3220, Acc train source: 84.06 , Acc train target: 91.17
D_loss train: 1.4011, G_los

/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.7226, Acc train source: 81.41 , Acc train target: 78.20
D_loss train: 1.6797, G_loss train: 0.8883
Test results - Loss: 0.720 - Acc: 0.703
Epoch: 20
C_loss train: 0.5679, Acc train source: 77.81 , Acc train target: 83.60
D_loss train: 1.4323, G_loss train: 0.7296
Test results - Loss: 0.710 - Acc: 0.631
Epoch: 30
C_loss train: 0.4902, Acc train source: 81.88 , Acc train target: 87.13
D_loss train: 1.3700, G_loss train: 0.6542
Test results - Loss: 1.019 - Acc: 0.642
Epoch: 40
C_loss train: 0.4270, Acc train source: 83.44 , Acc train target: 89.25
D_loss train: 1.3041, G_loss train: 0.5943
Test results - Loss: 0.799 - Acc: 0.673
Epoch: 50
C_loss train: 0.4798, Acc train source: 79.06 , Acc train target: 90.08
D_loss train: 1.3372, G_loss train: 0.6504
Test results - Loss: 1.103 - Acc: 0.682
Epoch: 60
C_loss train: 0.4006, Acc train source: 85.47 , Acc train target: 91.20
D_loss train: 1.3021, G_loss train: 0.5708
Test results - Loss: 1.369 - Acc: 0.563
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.6782, Acc train source: 80.45 , Acc train target: 76.10
D_loss train: 1.5761, G_loss train: 0.8379
Test results - Loss: 0.826 - Acc: 0.663
Epoch: 20
C_loss train: 0.5286, Acc train source: 82.86 , Acc train target: 82.20
D_loss train: 1.4398, G_loss train: 0.6817
Test results - Loss: 0.612 - Acc: 0.757
Epoch: 30
C_loss train: 0.3860, Acc train source: 85.54 , Acc train target: 85.90
D_loss train: 1.3618, G_loss train: 0.5442
Test results - Loss: 0.998 - Acc: 0.730
Epoch: 40
C_loss train: 0.3576, Acc train source: 84.82 , Acc train target: 88.25
D_loss train: 1.3248, G_loss train: 0.5185
Test results - Loss: 0.674 - Acc: 0.752
Epoch: 50
C_loss train: 0.3417, Acc train source: 85.18 , Acc train target: 89.58
D_loss train: 1.3332, G_loss train: 0.5024
Test results - Loss: 0.674 - Acc: 0.774
Epoch: 60
C_loss train: 0.3735, Acc train source: 83.75 , Acc train target: 90.67
D_loss train: 1.3304, G_loss train: 0.5340
Test results - Loss: 0.602 - Acc: 0.790
Epoch: 70


...

Epoch: 10
C_loss train: 0.6646, Acc train source: 80.05 , Acc train target: 79.45
D_loss train: 1.4632, G_loss train: 0.8187
Test results - Loss: 0.683 - Acc: 0.684
Epoch: 20
C_loss train: 0.4198, Acc train source: 83.70 , Acc train target: 85.08
D_loss train: 1.3949, G_loss train: 0.5682
Test results - Loss: 0.649 - Acc: 0.744
Epoch: 30
C_loss train: 0.4080, Acc train source: 83.65 , Acc train target: 87.52
D_loss train: 1.3597, G_loss train: 0.5584
Test results - Loss: 0.696 - Acc: 0.761
Epoch: 40
C_loss train: 0.3526, Acc train source: 84.33 , Acc train target: 89.11
D_loss train: 1.3321, G_loss train: 0.5074
Test results - Loss: 0.660 - Acc: 0.697
Epoch: 50
C_loss train: 0.3365, Acc train source: 85.14 , Acc train target: 90.25
D_loss train: 1.3152, G_loss train: 0.4962
Test results - Loss: 0.608 - Acc: 0.779
Epoch: 60
C_loss train: 0.2840, Acc train source: 87.12 , Acc train target: 91.27
D_loss train: 1.3073, G_loss train: 0.4458
Test results - Loss: 0.634 - Acc: 0.775
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.6261, Acc train source: 83.29 , Acc train target: 79.37
D_loss train: 1.4675, G_loss train: 0.7786
Test results - Loss: 0.571 - Acc: 0.777
Epoch: 20
C_loss train: 0.4278, Acc train source: 85.30 , Acc train target: 84.32
D_loss train: 1.3698, G_loss train: 0.5789
Test results - Loss: 0.547 - Acc: 0.800
Epoch: 30
C_loss train: 0.3584, Acc train source: 87.35 , Acc train target: 86.90
D_loss train: 1.3305, G_loss train: 0.5136
Test results - Loss: 0.645 - Acc: 0.749
Epoch: 40
C_loss train: 0.3134, Acc train source: 88.39 , Acc train target: 88.80
D_loss train: 1.3254, G_loss train: 0.4715
Test results - Loss: 0.521 - Acc: 0.810
Epoch: 50
C_loss train: 0.2889, Acc train source: 88.82 , Acc train target: 90.03
D_loss train: 1.3070, G_loss train: 0.4496
Test results - Loss: 0.605 - Acc: 0.792
Epoch: 60
C_loss train: 0.2626, Acc train source: 89.16 , Acc train target: 91.09
D_loss train: 1.3225, G_loss train: 0.4230
Test results - Loss: 0.605 - Acc: 0.792
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.6331, Acc train source: 83.57 , Acc train target: 77.48
D_loss train: 1.4223, G_loss train: 0.7829
Test results - Loss: 0.487 - Acc: 0.788
Epoch: 20
C_loss train: 0.4644, Acc train source: 86.67 , Acc train target: 82.10
D_loss train: 1.3343, G_loss train: 0.6190
Test results - Loss: 0.542 - Acc: 0.784
Epoch: 30
C_loss train: 0.4110, Acc train source: 86.95 , Acc train target: 84.68
D_loss train: 1.3078, G_loss train: 0.5740
Test results - Loss: 0.540 - Acc: 0.781
Epoch: 40
C_loss train: 0.3627, Acc train source: 88.31 , Acc train target: 86.19
D_loss train: 1.3348, G_loss train: 0.5212
Test results - Loss: 0.738 - Acc: 0.649
Epoch: 50
C_loss train: 0.3288, Acc train source: 88.98 , Acc train target: 87.54
D_loss train: 1.3079, G_loss train: 0.4920
Test results - Loss: 0.538 - Acc: 0.783
Epoch: 60
C_loss train: 0.3227, Acc train source: 89.39 , Acc train target: 88.52
D_loss train: 1.2931, G_loss train: 0.4911
Test results - Loss: 0.463 - Acc: 0.810
Epoch: 70


### target train aug target test sep

In [47]:
# Merge the datasets for source malware from March, April, and May
source_malware = merge_dataset(malware_march, malware_april)
source_malware = merge_dataset(source_malware,  malware_may)
# Merge the datasets for target malware from August(train) and September (test)
target_malware_train = malware_aug
target_malware_test = malware_sep



# convert it to binary labels
target_malware_train = binary_label(target_malware_train, True)
target_malware_test = binary_label(target_malware_test, True)
source_malware  = binary_label(source_malware, True)
target_normal = binary_label(target_normal, False)
source_normal = binary_label(source_normal, False)

print("Target malware train size: {}".format(len(target_malware_train)))
print("Target malware test size: {}".format(len(target_malware_test)))
print("Source malware size: {}".format(len(source_malware)))
print("Source normal size: {}".format(len(source_normal)))
           
           

Target malware train size: 1636
Target malware test size: 1300
Source malware size: 4033
Source normal size: 6510


In [48]:
# Split the target normal dataset into training and testing sets
target_normal_train, target_normal_ = train_test_split(target_normal, 0.33)
target_normal_test, _ = train_test_split(target_normal_, 0.5)
print(len(target_normal_train))
print(len(target_normal_test))

1903
1932


In [49]:
# Merge the source normal and source malware datasets
source = merge_dataset(source_normal, source_malware)

# Split the source dataset into training and testing sets
source_train, source_test = train_test_split(source, 0.75)
# Construct target training and testing sets
target_train = merge_dataset(target_normal_train, target_malware_train)
target_test = merge_dataset(target_normal_test, target_malware_test)



print("Target train dataset size: {}".format(len(target_train)))
print("Target test dataset size: {}".format(len(target_test)))
print("Source train dataset size: {}".format(len(source_train)))
print("Source test dataset size: {}".format((len(source_test))))

Target train dataset size: 3539
Target test dataset size: 3232
Source train dataset size: 7907
Source test dataset size: 2636


In [50]:
samples = [20, 50, 100, 200, 300, 500]

channels = 128  # Hidden units
layers = 3  # GIN layers
epochs = 80  # Number of training epochs
batch_size = 16  # Batch size
n_out = target_train.n_labels

for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))
    

    target_train_select, target_train_re = subsample(target_train, size)

    loader_source_tr = DisjointLoader(source_train, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_source_te = DisjointLoader(source_test,  batch_size=batch_size)


    loader_target_tr = DisjointLoader(target_train_select, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_target_te = DisjointLoader(target_test, batch_size=batch_size)

    # build model
    GIN = GIN0(channels, layers)


    model =DANN_GIN(loader_source_tr, loader_target_tr, loader_target_te, GIN, n_out, epochs==80)


    G, C = model.train()
    
    
    ################################################################################
    # Evaluate model
    ################################################################################
    results = []
    step = 0
    
    while step < loader_target_te.steps_per_epoch:      
        step += 1
        inputs, target = loader_target_te.__next__()
        pred = C(G(inputs, training=False), training=False)
        results.append(
            (
                f1_score(np.argmax(target, axis=1), np.argmax(pred, axis=1), average='weighted')
            )
        )
    print("Done. Test f1: {}".format(np.mean(results, 0)))
    
        
       

--------------------------Sample size 20-------------------------


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.7141, Acc train source: 70.00 , Acc train target: 83.00
D_loss train: 1.7167, G_loss train: 0.8808
Test results - Loss: 0.824 - Acc: 0.600
Epoch: 20
C_loss train: 0.5474, Acc train source: 79.38 , Acc train target: 88.50
D_loss train: 1.5475, G_loss train: 0.7046
Test results - Loss: 1.071 - Acc: 0.543
Epoch: 30
C_loss train: 0.5299, Acc train source: 71.25 , Acc train target: 90.33
D_loss train: 1.4865, G_loss train: 0.6877
Test results - Loss: 0.883 - Acc: 0.643
Epoch: 40
C_loss train: 0.5583, Acc train source: 72.50 , Acc train target: 91.12
D_loss train: 1.4256, G_loss train: 0.7176
Test results - Loss: 0.774 - Acc: 0.660
Epoch: 50
C_loss train: 0.3739, Acc train source: 83.12 , Acc train target: 92.70
D_loss train: 1.3804, G_loss train: 0.5358
Test results - Loss: 1.070 - Acc: 0.674
Epoch: 60
C_loss train: 0.4981, Acc train source: 80.62 , Acc train target: 93.08
D_loss train: 1.3348, G_loss train: 0.6630
Test results - Loss: 1.233 - Acc: 0.605
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.7940, Acc train source: 77.34 , Acc train target: 72.60
D_loss train: 1.5101, G_loss train: 0.9571
Test results - Loss: 0.828 - Acc: 0.666
Epoch: 20
C_loss train: 0.6082, Acc train source: 78.75 , Acc train target: 80.10
D_loss train: 1.4158, G_loss train: 0.7656
Test results - Loss: 1.128 - Acc: 0.702
Epoch: 30
C_loss train: 0.5015, Acc train source: 83.12 , Acc train target: 83.93
D_loss train: 1.3719, G_loss train: 0.6609
Test results - Loss: 1.003 - Acc: 0.729
Epoch: 40
C_loss train: 0.3907, Acc train source: 86.88 , Acc train target: 86.55
D_loss train: 1.3560, G_loss train: 0.5497
Test results - Loss: 1.306 - Acc: 0.631
Epoch: 50
C_loss train: 0.7239, Acc train source: 77.19 , Acc train target: 87.24
D_loss train: 1.3539, G_loss train: 0.8845
Test results - Loss: 0.720 - Acc: 0.705
Epoch: 60
C_loss train: 0.5093, Acc train source: 83.28 , Acc train target: 88.13
D_loss train: 1.3281, G_loss train: 0.6714
Test results - Loss: 0.860 - Acc: 0.691
Epoch: 70


...

Epoch: 10
C_loss train: 0.6612, Acc train source: 81.49 , Acc train target: 78.50
D_loss train: 1.5044, G_loss train: 0.8130
Test results - Loss: 0.864 - Acc: 0.718
Epoch: 20
C_loss train: 0.4734, Acc train source: 83.75 , Acc train target: 83.42
D_loss train: 1.3950, G_loss train: 0.6191
Test results - Loss: 0.808 - Acc: 0.720
Epoch: 30
C_loss train: 0.4263, Acc train source: 84.81 , Acc train target: 86.07
D_loss train: 1.3642, G_loss train: 0.5755
Test results - Loss: 0.764 - Acc: 0.704
Epoch: 40
C_loss train: 0.3437, Acc train source: 85.92 , Acc train target: 88.05
D_loss train: 1.3474, G_loss train: 0.4960
Test results - Loss: 0.645 - Acc: 0.764
Epoch: 50
C_loss train: 0.2934, Acc train source: 86.30 , Acc train target: 89.73
D_loss train: 1.3539, G_loss train: 0.4444
Test results - Loss: 1.024 - Acc: 0.648
Epoch: 60
C_loss train: 0.2576, Acc train source: 88.03 , Acc train target: 90.97
D_loss train: 1.3539, G_loss train: 0.4079
Test results - Loss: 0.708 - Acc: 0.770
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.6308, Acc train source: 82.17 , Acc train target: 80.00
D_loss train: 1.4544, G_loss train: 0.7824
Test results - Loss: 0.687 - Acc: 0.684
Epoch: 20
C_loss train: 0.4324, Acc train source: 86.05 , Acc train target: 84.65
D_loss train: 1.3625, G_loss train: 0.5840
Test results - Loss: 0.627 - Acc: 0.738
Epoch: 30
C_loss train: 0.3881, Acc train source: 86.36 , Acc train target: 86.89
D_loss train: 1.3549, G_loss train: 0.5398
Test results - Loss: 0.998 - Acc: 0.757
Epoch: 40
C_loss train: 0.3605, Acc train source: 87.17 , Acc train target: 88.22
D_loss train: 1.3615, G_loss train: 0.5121
Test results - Loss: 0.755 - Acc: 0.655
Epoch: 50
C_loss train: 0.2987, Acc train source: 87.60 , Acc train target: 89.55
D_loss train: 1.3453, G_loss train: 0.4521
Test results - Loss: 0.740 - Acc: 0.702
Epoch: 60
C_loss train: 0.2824, Acc train source: 88.80 , Acc train target: 90.42
D_loss train: 1.3530, G_loss train: 0.4339
Test results - Loss: 0.861 - Acc: 0.707
Epoch: 70


/home/li3944/.local/lib/python3.9/site-packages/spektral/data/utils.py:221: UserWarning: you are shuffling a 'GraphData' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Epoch: 10
C_loss train: 0.6676, Acc train source: 82.44 , Acc train target: 78.08
D_loss train: 1.4188, G_loss train: 0.8206
Test results - Loss: 0.493 - Acc: 0.795
Epoch: 20
C_loss train: 0.4616, Acc train source: 87.14 , Acc train target: 82.65
D_loss train: 1.3564, G_loss train: 0.6134
Test results - Loss: 1.015 - Acc: 0.653
Epoch: 30
C_loss train: 0.3868, Acc train source: 87.07 , Acc train target: 85.38
D_loss train: 1.3442, G_loss train: 0.5404
Test results - Loss: 0.475 - Acc: 0.812
Epoch: 40
C_loss train: 0.3551, Acc train source: 88.08 , Acc train target: 87.05
D_loss train: 1.3282, G_loss train: 0.5137
Test results - Loss: 0.729 - Acc: 0.739
Epoch: 50
C_loss train: 0.3277, Acc train source: 88.39 , Acc train target: 88.14
D_loss train: 1.3505, G_loss train: 0.4806
Test results - Loss: 0.529 - Acc: 0.796
Epoch: 60
C_loss train: 0.2884, Acc train source: 89.51 , Acc train target: 89.10
D_loss train: 1.3296, G_loss train: 0.4447
Test results - Loss: 0.520 - Acc: 0.806
Epoch: 70
